# Module 5 — Notebook 02: DEAP GA Feature Selector

## Learning Objectives

By the end of this notebook you will:
1. Implement the same GA from Notebook 01 using the DEAP framework
2. Build a custom fitness function with parsimony pressure inside DEAP
3. Use DEAP's Hall of Fame and Statistics Logbook
4. Visualise the Pareto front (accuracy vs feature count)
5. Benchmark GA vs Boruta vs Lasso vs mRMR on the same dataset
6. Demonstrate DEAP's parallelisation capabilities

**Estimated time**: 15 minutes  
**Dataset**: UCI Breast Cancer (569 samples, 30 features)  
**Requires**: `pip install deap boruta` (mRMR via `pip install mrmr-selection`)

---

## 1. Setup

In [ ]:
# Install dependencies if needed
import subprocess, sys

def install_if_missing(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

install_if_missing('deap')
install_if_missing('boruta')

# Try mrmr-selection; fall back gracefully if unavailable
try:
    install_if_missing('mrmr-selection', 'mrmr')
    HAS_MRMR = True
except Exception:
    HAS_MRMR = False
    print("mrmr-selection not available — will skip mRMR comparison")

print("Dependencies ready.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import time

from deap import base, creator, tools, algorithms
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (cross_val_score, train_test_split,
                                     StratifiedKFold)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries loaded.")

### 1.1 Load Data

In [ ]:
# Purpose: Prepare data — identical split to Notebook 01 for fair comparison
data = load_breast_cancer()
feature_names = list(data.feature_names)
X_raw = data.data
y = data.target

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# Also keep as DataFrame for mRMR
X_train_df = pd.DataFrame(X_train, columns=feature_names)
y_train_s = pd.Series(y_train)

N_FEATURES = X_train.shape[1]
print(f"Train: {X_train.shape}, Test: {X_test.shape}, Features: {N_FEATURES}")

---
## 2. DEAP Setup

In [ ]:
# Purpose: Register DEAP types, operators, and fitness function
# Key Concept: DEAP uses a creator/toolbox pattern to configure the GA

# ── Fitness and Individual types ────────────────────────────────────────────────
# DEAP requires creating these types once per Python session
# Guard against re-registration in Jupyter re-runs
if 'FitnessMax' not in dir(creator):
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
if 'Individual' not in dir(creator):
    creator.create("Individual", list, fitness=creator.FitnessMax)

# ── Toolbox ────────────────────────────────────────────────────────────────────
toolbox = base.Toolbox()

# Chromosome: list of 0/1 integers
toolbox.register("attr_bool", np.random.randint, 0, 2)
toolbox.register("individual", tools.initRepeat,
                 creator.Individual, toolbox.attr_bool, N_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

print("DEAP toolbox configured.")

# Quick sanity check
test_ind = toolbox.individual()
print(f"Sample individual (first 10 genes): {test_ind[:10]}")
print(f"Individual type: {type(test_ind)}")

---
## 3. Custom Fitness Function with Parsimony Pressure

In [ ]:
# Purpose: Define fitness with parsimony inside DEAP's required signature
# Key Concept: DEAP fitness functions must return a tuple — (value,) for single-objective

PARSIMONY_WEIGHT = 0.01
_deap_cache: dict[bytes, float] = {}


def deap_evaluate(individual):
    """
    DEAP-compatible fitness: stratified 5-fold CV accuracy − parsimony penalty.

    Parameters
    ----------
    individual : DEAP Individual (list of 0/1)

    Returns
    -------
    tuple[float]
        Single-element tuple required by DEAP.
    """
    chrom = np.array(individual, dtype=np.int8)
    key = chrom.tobytes()

    if key in _deap_cache:
        return (_deap_cache[key],)

    selected = np.where(chrom == 1)[0]
    if len(selected) == 0:
        _deap_cache[key] = -1.0
        return (-1.0,)

    try:
        model = LogisticRegression(max_iter=500, random_state=0)
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(
            model, X_train[:, selected], y_train,
            cv=skf, scoring='accuracy'
        )
        cv_acc = float(scores.mean())
    except Exception:
        _deap_cache[key] = -1.0
        return (-1.0,)

    penalty = PARSIMONY_WEIGHT * sum(individual) / N_FEATURES
    fitness = cv_acc - penalty
    _deap_cache[key] = fitness
    return (fitness,)


# Register operators
toolbox.register("evaluate", deap_evaluate)
toolbox.register("mate", tools.cxUniform, indpb=0.5)          # uniform crossover
toolbox.register("mutate", tools.mutFlipBit, indpb=1.0/N_FEATURES)  # bit-flip
toolbox.register("select", tools.selTournament, tournsize=3)   # tournament k=3

# Test fitness
sample = toolbox.individual()
fit_val = deap_evaluate(sample)
print(f"Sample fitness: {fit_val[0]:.4f} (selected {sum(sample)}/{N_FEATURES} features)")

---
## 4. Hall of Fame and Statistics

In [ ]:
# Purpose: Configure DEAP monitoring: Hall of Fame + Statistics + Logbook
# Key Concept: Hall of Fame tracks the best N individuals ever seen across all generations

# Hall of Fame: stores top 10 unique individuals ever observed
hof = tools.HallOfFame(10)

# Statistics: computed on fitness values each generation
stats = tools.Statistics(lambda ind: ind.fitness.values[0])
stats.register("max", np.max)
stats.register("mean", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)

print("Hall of Fame (size 10) and Statistics configured.")
print("Tracked statistics per generation: max, mean, std, min")

---
## 5. Run DEAP GA

In [ ]:
# Purpose: Run the DEAP GA using eaSimple algorithm
# Key Concept: eaSimple = standard generational GA with configurable operators

POP_SIZE = 50
N_GENERATIONS = 80
CROSSOVER_RATE = 0.8
MUTATION_RATE = 1.0   # DEAP mutpb = per-individual rate; actual bit-flip rate is indpb=1/p

# Fresh population
population = toolbox.population(n=POP_SIZE)

print(f"Running DEAP GA: {POP_SIZE} individuals, {N_GENERATIONS} generations...")
t0 = time.time()

population, logbook = algorithms.eaSimple(
    population,
    toolbox,
    cxpb=CROSSOVER_RATE,
    mutpb=MUTATION_RATE,
    ngen=N_GENERATIONS,
    stats=stats,
    halloffame=hof,
    verbose=False,  # set True to see per-generation output
)

elapsed = time.time() - t0
print(f"Done in {elapsed:.1f}s | Cache size: {len(_deap_cache)} unique evaluations")

# Best solution
best_deap = hof[0]
best_selected = [i for i, g in enumerate(best_deap) if g == 1]
print(f"\nBest solution: {len(best_selected)} features, fitness={best_deap.fitness.values[0]:.4f}")
print(f"Top 5 Hall of Fame entries:")
for i, ind in enumerate(hof[:5]):
    n = sum(ind)
    print(f"  [{i+1}] {n} features, fitness={ind.fitness.values[0]:.4f}")

---
## 6. Visualise Convergence and Pareto Front

In [ ]:
# Purpose: Plot convergence curves from DEAP logbook + Pareto front (accuracy vs features)
# Key Concept: Pareto front shows the accuracy-complexity tradeoff across all evaluated subsets

# Extract logbook data
gen = logbook.select('gen')
fit_max = logbook.select('max')
fit_mean = logbook.select('mean')
fit_std = logbook.select('std')

# Build Pareto front from all cached evaluations
# cache maps chromosome_bytes → fitness
# We need to reconstruct (n_features, accuracy) pairs
pareto_data = []
for chrom_bytes, fitness in _deap_cache.items():
    if fitness > 0:  # skip penalised
        chrom = np.frombuffer(chrom_bytes, dtype=np.int8)
        n_sel = int(chrom.sum())
        # Approximate accuracy from fitness (reverse parsimony)
        approx_acc = fitness + PARSIMONY_WEIGHT * n_sel / N_FEATURES
        pareto_data.append((n_sel, approx_acc, fitness))

pareto_df = pd.DataFrame(pareto_data, columns=['n_features', 'accuracy', 'fitness'])

# Compute true Pareto front (non-dominated solutions)
def is_pareto_optimal(df, col_max1, col_min2):
    """Return boolean mask of Pareto-optimal rows (maximise col1, minimise col2)."""
    is_optimal = np.ones(len(df), dtype=bool)
    acc = df[col_max1].values
    n = df[col_min2].values
    for i in range(len(df)):
        # i is dominated if there exists j such that acc[j] >= acc[i] AND n[j] <= n[i]
        # with at least one strict inequality
        dominated = np.any(
            (acc >= acc[i]) & (n <= n[i]) &
            ((acc > acc[i]) | (n < n[i]))
        )
        if dominated:
            is_optimal[i] = False
    return is_optimal

if len(pareto_df) > 0:
    pareto_mask = is_pareto_optimal(pareto_df, 'accuracy', 'n_features')
    pareto_front = pareto_df[pareto_mask].sort_values('n_features')
else:
    pareto_front = pd.DataFrame(columns=['n_features', 'accuracy', 'fitness'])

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Fitness convergence
axes[0].plot(gen, fit_max, 'r-', lw=2, label='Best')
axes[0].plot(gen, fit_mean, 'b--', lw=1.5, alpha=0.7, label='Mean')
axes[0].fill_between(
    gen,
    np.array(fit_mean) - np.array(fit_std),
    np.array(fit_mean) + np.array(fit_std),
    alpha=0.2, color='blue', label='±1 std'
)
axes[0].set_xlabel('Generation')
axes[0].set_ylabel('Fitness')
axes[0].set_title('DEAP Fitness Convergence')
axes[0].legend()

# Panel 2: Pareto front
if len(pareto_df) > 0:
    axes[1].scatter(pareto_df['n_features'], pareto_df['accuracy'],
                    alpha=0.3, s=15, color='lightblue', label='All evaluated')
    axes[1].scatter(pareto_front['n_features'], pareto_front['accuracy'],
                    color='red', s=60, zorder=5, label='Pareto front')
    if len(pareto_front) > 1:
        pf_sorted = pareto_front.sort_values('n_features')
        axes[1].plot(pf_sorted['n_features'], pf_sorted['accuracy'],
                     'r-', alpha=0.5, lw=1.5)
    # Mark best solution
    axes[1].scatter([len(best_selected)], 
                    [best_deap.fitness.values[0] + PARSIMONY_WEIGHT * len(best_selected) / N_FEATURES],
                    color='gold', s=200, marker='*', zorder=6, label='Best (weighted)')
axes[1].set_xlabel('Number of Features')
axes[1].set_ylabel('CV Accuracy')
axes[1].set_title('Pareto Front: Accuracy vs Feature Count')
axes[1].legend(fontsize=8)

# Panel 3: Hall of Fame — feature count distribution
hof_counts = [sum(ind) for ind in hof]
hof_fits = [ind.fitness.values[0] for ind in hof]
sc = axes[2].scatter(hof_counts, hof_fits, c=range(len(hof)),
                     cmap='coolwarm', s=100, zorder=5)
plt.colorbar(sc, ax=axes[2], label='Hall of Fame rank')
for i, (n, f) in enumerate(zip(hof_counts, hof_fits)):
    axes[2].annotate(f'#{i+1}', (n, f), textcoords='offset points',
                     xytext=(4, 4), fontsize=8)
axes[2].set_xlabel('Number of Features')
axes[2].set_ylabel('Fitness')
axes[2].set_title('Hall of Fame Top 10')

plt.tight_layout()
plt.show()

print(f"Pareto front contains {len(pareto_front)} non-dominated solutions")
print(f"Accuracy range: {pareto_front['accuracy'].min():.4f} – {pareto_front['accuracy'].max():.4f}")
print(f"Feature count range: {pareto_front['n_features'].min()} – {pareto_front['n_features'].max()}")

---
## 7. Benchmark: GA vs Boruta vs Lasso vs mRMR

In [ ]:
# Purpose: Compare GA against three alternative feature selection methods
# Key Concept: Benchmark on identical train/test split, same evaluation model

results = {}

# ── 1. GA (DEAP) ───────────────────────────────────────────────────────────────
ga_idx = np.array(best_selected)
model = LogisticRegression(max_iter=500, random_state=0)
model.fit(X_train[:, ga_idx], y_train)
results['GA (DEAP)'] = {
    'n_features': len(ga_idx),
    'test_acc': model.score(X_test[:, ga_idx], y_test),
    'cv_acc': cross_val_score(model, X_train[:, ga_idx], y_train,
                              cv=5, scoring='accuracy').mean(),
    'indices': ga_idx,
}
print(f"GA (DEAP): {results['GA (DEAP)']['n_features']} features, "
      f"test_acc={results['GA (DEAP)']['test_acc']:.4f}")

# ── 2. Boruta ──────────────────────────────────────────────────────────────────
try:
    from boruta import BorutaPy

    rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
    boruta = BorutaPy(rf, n_estimators='auto', verbose=0, random_state=42,
                      max_iter=30)  # reduced for speed
    boruta.fit(X_train, y_train)
    boruta_idx = np.where(boruta.support_)[0]

    if len(boruta_idx) > 0:
        model.fit(X_train[:, boruta_idx], y_train)
        results['Boruta'] = {
            'n_features': len(boruta_idx),
            'test_acc': model.score(X_test[:, boruta_idx], y_test),
            'cv_acc': cross_val_score(model, X_train[:, boruta_idx], y_train,
                                      cv=5, scoring='accuracy').mean(),
            'indices': boruta_idx,
        }
        print(f"Boruta: {results['Boruta']['n_features']} features, "
              f"test_acc={results['Boruta']['test_acc']:.4f}")
    else:
        print("Boruta selected 0 features — skipping")
except Exception as e:
    print(f"Boruta error: {e}")

In [ ]:
# ── 3. Lasso (L1 regularisation) ───────────────────────────────────────────────
from sklearn.linear_model import LogisticRegressionCV

lasso = LogisticRegressionCV(
    Cs=np.logspace(-3, 1, 20),
    penalty='l1', solver='saga',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy', max_iter=2000, random_state=42, n_jobs=-1
)
lasso.fit(X_train, y_train)

lasso_idx = np.where(np.abs(lasso.coef_[0]) > 1e-4)[0]
if len(lasso_idx) == 0:
    lasso_idx = np.array([np.argmax(np.abs(lasso.coef_[0]))])  # at least 1

model_lasso = LogisticRegression(max_iter=500, random_state=0)
model_lasso.fit(X_train[:, lasso_idx], y_train)
results['Lasso (L1)'] = {
    'n_features': len(lasso_idx),
    'test_acc': model_lasso.score(X_test[:, lasso_idx], y_test),
    'cv_acc': cross_val_score(model_lasso, X_train[:, lasso_idx], y_train,
                              cv=5, scoring='accuracy').mean(),
    'indices': lasso_idx,
}
print(f"Lasso (L1): {results['Lasso (L1)']['n_features']} features, "
      f"test_acc={results['Lasso (L1)']['test_acc']:.4f}")

# ── 4. mRMR ─────────────────────────────────────────────────────────────────────
if HAS_MRMR:
    try:
        import mrmr
        n_select = len(ga_idx)  # same count as GA for fair comparison
        mrmr_selected = mrmr.mrmr_classif(
            X=X_train_df, y=y_train_s, K=n_select
        )
        mrmr_idx = np.array([feature_names.index(f) for f in mrmr_selected
                             if f in feature_names])
        if len(mrmr_idx) > 0:
            model_mrmr = LogisticRegression(max_iter=500, random_state=0)
            model_mrmr.fit(X_train[:, mrmr_idx], y_train)
            results['mRMR'] = {
                'n_features': len(mrmr_idx),
                'test_acc': model_mrmr.score(X_test[:, mrmr_idx], y_test),
                'cv_acc': cross_val_score(model_mrmr, X_train[:, mrmr_idx], y_train,
                                          cv=5, scoring='accuracy').mean(),
                'indices': mrmr_idx,
            }
            print(f"mRMR: {results['mRMR']['n_features']} features, "
                  f"test_acc={results['mRMR']['test_acc']:.4f}")
    except Exception as e:
        print(f"mRMR error: {e}")

# ── 5. All features baseline ────────────────────────────────────────────────────
model_all = LogisticRegression(max_iter=500, random_state=0)
model_all.fit(X_train, y_train)
results['All Features'] = {
    'n_features': N_FEATURES,
    'test_acc': model_all.score(X_test, y_test),
    'cv_acc': cross_val_score(model_all, X_train, y_train,
                              cv=5, scoring='accuracy').mean(),
    'indices': np.arange(N_FEATURES),
}
print(f"All Features: {results['All Features']['n_features']} features, "
      f"test_acc={results['All Features']['test_acc']:.4f}")

In [ ]:
# Purpose: Visualise the benchmark comparison
# Key Concept: Scatter plot of accuracy vs feature count captures the full trade-off

methods = list(results.keys())
n_feats = [results[m]['n_features'] for m in methods]
test_accs = [results[m]['test_acc'] for m in methods]
cv_accs = [results[m]['cv_acc'] for m in methods]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Test accuracy vs feature count
colors = plt.cm.tab10(np.linspace(0, 1, len(methods)))
for i, method in enumerate(methods):
    axes[0].scatter(n_feats[i], test_accs[i], s=150, color=colors[i],
                    label=method, zorder=5, edgecolors='black', linewidths=0.5)
    axes[0].annotate(
        f" {method}\n ({n_feats[i]} feats)",
        (n_feats[i], test_accs[i]),
        fontsize=8, va='bottom'
    )
axes[0].set_xlabel('Number of Selected Features')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Test Accuracy vs Feature Count')
axes[0].legend(fontsize=8)

# Panel 2: Side-by-side bars
x = np.arange(len(methods))
width = 0.35
bars1 = axes[1].bar(x - width/2, cv_accs, width, label='CV Accuracy', color='steelblue', alpha=0.8)
bars2 = axes[1].bar(x + width/2, test_accs, width, label='Test Accuracy', color='orange', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods, rotation=15, ha='right')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.88, 1.0)
axes[1].set_title('CV vs Test Accuracy by Method')
axes[1].legend()
# Annotate with feature counts
for bar, n in zip(bars1, n_feats):
    axes[1].text(bar.get_x() + bar.get_width()/2, 0.889, f'k={n}',
                 ha='center', va='bottom', fontsize=8, color='navy')

plt.tight_layout()
plt.show()

# Summary table
print("\n" + "=" * 65)
print(f"{'Method':<20} {'Features':>10} {'CV Acc':>10} {'Test Acc':>10}")
print("=" * 65)
for method in methods:
    r = results[method]
    print(f"{method:<20} {r['n_features']:>10} {r['cv_acc']:>10.4f} {r['test_acc']:>10.4f}")
print("=" * 65)

---
## 8. DEAP Parallelisation

In [ ]:
# Purpose: Demonstrate DEAP's multiprocessing capability
# Key Concept: Replace toolbox.map with pool.map for parallel population evaluation

import multiprocessing

print("CPU cores available:", multiprocessing.cpu_count())

# Define a standalone fitness function for pickling (required by multiprocessing)
# Note: X_train and y_train must be in module scope for pickling to work
_X_mp = X_train.copy()
_y_mp = y_train.copy()
_N_mp = N_FEATURES
_PW_mp = PARSIMONY_WEIGHT


def mp_evaluate(individual):
    """Picklable fitness function for multiprocessing pool."""
    chrom = np.array(individual, dtype=np.int8)
    selected = np.where(chrom == 1)[0]
    if len(selected) == 0:
        return (-1.0,)
    try:
        model = LogisticRegression(max_iter=500, random_state=0)
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_val_score(
            model, _X_mp[:, selected], _y_mp, cv=skf, scoring='accuracy'
        )
        fitness = float(scores.mean()) - _PW_mp * len(selected) / _N_mp
        return (fitness,)
    except Exception:
        return (-1.0,)


# Time sequential evaluation on small population
toolbox.register("evaluate_seq", deap_evaluate)
pop_small = toolbox.population(n=20)

t0 = time.time()
for ind in pop_small:
    ind.fitness.values = deap_evaluate(ind)
seq_time = time.time() - t0

# Time parallel evaluation
pop_parallel = toolbox.population(n=20)

if __name__ == '__main__' or True:  # True for Jupyter compatibility
    try:
        with multiprocessing.Pool(processes=min(4, multiprocessing.cpu_count())) as pool:
            toolbox_mp = base.Toolbox()
            toolbox_mp.register("map", pool.map)
            toolbox_mp.register("evaluate", mp_evaluate)

            t0 = time.time()
            fitnesses = list(toolbox_mp.map(mp_evaluate, pop_parallel))
            par_time = time.time() - t0

        speedup = seq_time / par_time if par_time > 0 else 1.0
        print(f"Sequential (20 individuals): {seq_time:.2f}s")
        print(f"Parallel   (20 individuals): {par_time:.2f}s")
        print(f"Speedup: {speedup:.1f}×")
        print("\nNote: speedup depends on CPU count and fitness function cost.")
        print("For fast fitness (LogReg with caching), overhead may dominate.")
        print("Speedup is larger for expensive models (XGBoost, neural nets).")
    except Exception as e:
        print(f"Parallel execution: {e}")
        print("(Multiprocessing may not work in all Jupyter environments)")
        print(f"Sequential time for reference: {seq_time:.2f}s for 20 individuals")

---
## 9. Summary

### DEAP vs Custom GA — What We Learned

| Aspect | Custom (Notebook 01) | DEAP (This Notebook) |
|:---|:---:|:---:|
| Lines of code | ~150 | ~60 |
| Operator control | Full | High (via register) |
| Statistics | Manual | Built-in Logbook |
| Hall of Fame | Manual | Built-in |
| Parallelism | Manual joblib | pool.map (3 lines) |
| Multi-objective | Requires extension | NSGA-II built in |
| Debugging | Easier | Moderate |

### Key Insights from the Benchmark

1. **GA vs Lasso**: Lasso uses L1 penalty which also promotes sparsity — on linearly-separable data like breast cancer, Lasso often matches GA with fewer features. GA advantage is larger on non-linear interactions.
2. **GA vs Boruta**: Boruta provides confirmed/tentative/rejected labels — conservative. May miss some useful features but rarely includes noise. GA is more aggressive.
3. **Pareto front**: Rather than a single solution, the Pareto front shows the complete accuracy-complexity tradeoff — useful when the acceptable number of features is flexible.
4. **Hall of Fame**: The top 10 solutions often have very similar feature sets — confirming which features are truly essential across multiple runs.

### Next Steps

- **Notebook 03**: Systematic parameter tuning — heatmaps of pop_size × mutation_rate, adaptive parameter strategies